In [3]:
# https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_train.csv

import pandas as pd

train = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_train.csv')
display(train.head(3))
display(train.tail(3))

,biz_type,area,age,num_households,gas_totl
0,public,93.291780,33,61,499
1,residential,97.509495,34,131,1156
2,residential,97.140181,5,30,392


,biz_type,area,age,num_households,gas_totl
157,industrial,76.334688,36,133,1157
158,residential,62.510623,37,121,1070
159,public,91.744768,31,26,387


In [4]:
test = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_test.csv')
display(test.head(3))
display(test.tail(3))
# test - 'gas_totl' -> X_test

,biz_type,area,age,num_households,gas_totl
0,industrial,86.256496,20,46,NaN
1,commercial,112.340076,31,112,NaN
2,industrial,116.323054,7,30,NaN


,biz_type,area,age,num_households,gas_totl
37,commercial,111.997933,46,18,NaN
38,industrial,124.604227,34,168,NaN
39,residential,53.638635,28,93,NaN


In [5]:
# 2. 탐색 -> isNaN(), isNull()
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   biz_type        160 non-null    str    
 1   area            160 non-null    float64
 2   age             160 non-null    int64  
 3   num_households  160 non-null    int64  
 4   gas_totl        160 non-null    int64  
dtypes: float64(1), int64(3), str(1)
memory usage: 6.4 KB


In [6]:
# y값을 탐색 -> 사용량이니까 0, - 값은 행을 삭제
(train['gas_totl'] <= 0).sum() # True/False 시리즈를 sum()집계 True:1, False:0 변경 후 집계

np.int64(20)

In [7]:
# 0작거나 같은 값이 20개 있다 -> 결정 : 삭제
# 노이즈 행 삭제후 train2 저장
train2 = train[train['gas_totl'] > 0]
train2

,biz_type,area,age,num_households,gas_totl
0,public,93.291780,33,61,499
1,residential,97.509495,34,131,1156
2,residential,97.140181,5,30,392
3,residential,65.789408,12,166,1343
4,mixed,92.631000,28,14,154
...,...,...,...,...,...
155,industrial,89.174291,23,94,851
156,public,97.130755,14,145,1335
157,industrial,76.334688,36,133,1157
158,residential,62.510623,37,121,1070


In [8]:
# train2를 X와 y 열분리

# X = train[['biz_type','area',	'age'	,'num_households']]
X = train.drop(columns=['gas_totl'])
# X = train.drop('gas_totl', axis=1)

y = train['gas_totl']

display(X.head())
display(y.head())

,biz_type,area,age,num_households
0,public,93.291780,33,61
1,residential,97.509495,34,131
2,residential,97.140181,5,30
3,residential,65.789408,12,166
4,mixed,92.631000,28,14


0     499
1    1156
2     392
3    1343
4     154
Name: gas_totl, dtype: int64

In [9]:
# 3. 데이터 분할 - 학습,검증(X_train, X_val, y_train, y_val), 예측(X_test)
from sklearn.model_selection import train_test_split
# help(train_test_split)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)
print(X_train.shape)
print(X_val.shape)

X_test = test.drop(columns=['gas_totl'])
print(X_test.columns)
display(X_test.head())

(128, 4)
(32, 4)
Index(['biz_type', 'area', 'age', 'num_households'], dtype='str')


,biz_type,area,age,num_households
0,industrial,86.256496,20,46
1,commercial,112.340076,31,112
2,industrial,116.323054,7,30
3,residential,86.620503,25,146
4,mixed,105.421235,17,14


In [10]:
# 4.전처리(스케일링, 인코딩)
# X(X_train, X_val, X_test)의 범주형 변수에 대해 인코딩
print(X_train.columns)
# 컬럼이 많아 타이핑하기가 힘들다. api사용
# -> select_dtypes('number').columns, select_dtypes('object').columns
# ['biz_type', 'area', 'age', 'num_households']

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').set_output(transform = 'pandas') # OneHotEncoder결과물을 pandas자료타입으로 반환하도록

Index(['biz_type', 'area', 'age', 'num_households'], dtype='str')


In [11]:
# transform = 'pandas' DataFrame타입으로 반환된다.(X_train_en, X_val_en, X_test_en)
X_train_en = encoder.fit_transform(X_train[['biz_type']])
X_val_en = encoder.transform(X_val[['biz_type']])
X_test_en = encoder.transform(X_test[['biz_type']])

# OneHotEncoder 전처리 변환 후 확인
display(X_train_en.head())
display(X_val_en.head())
display(X_test_en.head())


,biz_type_commercial,biz_type_industrial,biz_type_mixed,biz_type_public,biz_type_residential
111,0.0,1.0,0.0,0.0,0.0
131,0.0,0.0,0.0,1.0,0.0
58,1.0,0.0,0.0,0.0,0.0
102,0.0,0.0,0.0,1.0,0.0
47,0.0,1.0,0.0,0.0,0.0


,biz_type_commercial,biz_type_industrial,biz_type_mixed,biz_type_public,biz_type_residential
29,0.0,0.0,0.0,1.0,0.0
42,1.0,0.0,0.0,0.0,0.0
14,0.0,1.0,0.0,0.0,0.0
91,0.0,0.0,1.0,0.0,0.0
81,0.0,1.0,0.0,0.0,0.0


,biz_type_commercial,biz_type_industrial,biz_type_mixed,biz_type_public,biz_type_residential
0,0.0,1.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,1.0,0.0,0.0


In [12]:
# 인코딩 된 열로 최종 X를 변경
X_train_pre = pd.concat([X_train_en, X_train[['area', 'age', 'num_households']]], axis=1)
X_val_pre = pd.concat([X_val_en, X_val[['area', 'age', 'num_households']]], axis=1)
X_test_pre = pd.concat([X_test_en, X_test[['area', 'age', 'num_households']]], axis=1)

display(X_train_pre.head())
display(X_val_pre.head())
display(X_test_pre.head())

,biz_type_commercial,biz_type_industrial,biz_type_mixed,biz_type_public,biz_type_residential,area,age,num_households
111,0.0,1.0,0.0,0.0,0.0,110.559209,10,181
131,0.0,0.0,0.0,1.0,0.0,86.661133,16,22
58,1.0,0.0,0.0,0.0,0.0,103.048417,44,50
102,0.0,0.0,0.0,1.0,0.0,104.579571,21,70
47,0.0,1.0,0.0,0.0,0.0,111.337881,27,166


,biz_type_commercial,biz_type_industrial,biz_type_mixed,biz_type_public,biz_type_residential,area,age,num_households
29,0.0,0.0,0.0,1.0,0.0,113.981848,13,107
42,1.0,0.0,0.0,0.0,0.0,80.802107,1,184
14,0.0,1.0,0.0,0.0,0.0,91.321168,8,99
91,0.0,0.0,1.0,0.0,0.0,122.098564,22,145
81,0.0,1.0,0.0,0.0,0.0,99.047126,12,63


,biz_type_commercial,biz_type_industrial,biz_type_mixed,biz_type_public,biz_type_residential,area,age,num_households
0,0.0,1.0,0.0,0.0,0.0,86.256496,20,46
1,1.0,0.0,0.0,0.0,0.0,112.340076,31,112
2,0.0,1.0,0.0,0.0,0.0,116.323054,7,30
3,0.0,0.0,0.0,0.0,1.0,86.620503,25,146
4,0.0,0.0,1.0,0.0,0.0,105.421235,17,14


In [13]:
# 5. 모델 학습, 평가
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=1)

# 트래인 데이터 학습(X_train_pre, y_tarin)
model.fit(X_train_pre, y_train)

UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence

UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence

RandomForestRegressor(random_state=1)

In [14]:
# 예측 X_val_pre 입력 후 예측
val_pred = model.predict(X_val_pre)

# 평가(val_pred, y_val)
from sklearn.metrics import root_mean_squared_error, r2_score

RMSE = root_mean_squared_error(y_val, val_pred)
R2 = r2_score(y_val, val_pred)

print('RMSE:', RMSE)
print('R2:', R2)

RMSE: 231.41149195615805
R2: 0.7975116583104999


In [15]:
# 6. 모델 저장
import joblib
joblib.dump(model, 'model.joblib')

['model.joblib']

In [16]:
# 7. 저장된 모델을 이용하여 새로운 문제 예측
model2 = joblib.load('model.joblib')

test_pred = model2.predict(X_test_pre)
test_pred

array([ 487.56, 1005.32,  339.2 ,  950.37,  213.12,  192.24,  766.33,
        207.59, 1596.28, 1134.59,  234.44, 1292.22,  528.63, 1522.38,
        564.94, 1006.86,  759.36,  198.25,  928.42,  909.2 ,  710.6 ,
        449.81,  983.73,  239.97,  918.95, 1287.64,  137.04,  808.96,
       1012.81,  212.09,  225.46,  390.49, 1698.28, 1262.18,  266.57,
        462.15, 1325.19,  134.02, 1460.96,  316.44])

In [17]:
# 8. 보고서

result_df = pd.DataFrame(test_pred, columns=['예측값'])
display(result_df.head())
# csv
result_df.to_csv('result.csv', index=False)


,예측값
0,487.56
1,1005.32
2,339.20
3,950.37
4,213.12
